# E-commerce Sales Analysis

# ============================================
# 1. IMPORTS
# ============================================

In [ ]:
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt

# ============================================
# 2. DATA LOADING
# ============================================

In [ ]:
df = pd.read_csv("../data/Sample - Superstore.csv", encoding="latin1")

In [ ]:
print("Dataset carregado com sucesso.")
print(f"Shape: {df.shape}")

In [ ]:
df.head()

# ============================================
# 3. INITIAL AUDIT
# ============================================

In [ ]:
print("\n--- Info ---")
df.info()

print("\n--- Estatísticas descritivas (numéricas) ---")
df.describe()

print("\n--- Valores únicos por coluna ---")
print(df.nunique())

print("\n--- Duplicatas ---")
print(f"Linhas duplicadas: {df.duplicated().sum()}")

print("\n--- Categorias ---")
print(df["Category"].unique())

print("\n--- Descontos únicos ---")
print(sorted(df["Discount"].unique()))

# ============================================
# 4. DATA PREPARATION
# ============================================

In [ ]:
# Converter datas
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

# Features temporais
df["Order Year"] = df["Order Date"].dt.year
df["Order Month"] = df["Order Date"].dt.to_period("M")

# Dias de envio
df["Shipping Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

print("\nPreparação concluída.")
print(df[["Order Date", "Ship Date", "Order Year", "Order Month", "Shipping Days"]].head())

# ============================================
# 5. OVERALL SALES & PROFIT
# ============================================

In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
overall_margin = total_profit / total_sales

print(f"Total Sales: ${total_sales:,.2f}")
print(f"Total Profit: ${total_profit:,.2f}")
print(f"Overall Margin: {overall_margin:.2%}")

# ============================================
# 6. CATEGORY ANALYSIS
# ============================================

In [ ]:
category = (
    df.groupby("Category")[["Sales", "Profit"]]
    .sum()
    .assign(Margin=lambda x: x["Profit"] / x["Sales"])
    .sort_values("Profit", ascending=False)
)
print("\n--- Category Performance ---")
print(category)

# Gráfico melhorado - Lucro por Categoria
fig, ax = plt.subplots(figsize=(8, 5))
category["Profit"].plot(kind="bar", ax=ax, color=["#2ecc71", "#3498db", "#e74c3c"])
ax.set_title("Profit by Category", fontsize=14, pad=15)
ax.set_ylabel("Profit ($)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
ax.axhline(0, color="black", linewidth=0.8)  # linha zero para referência
plt.tight_layout()
plt.show()

# ============================================
# 7. DISCOUNT ANALYSIS
# ============================================

In [ ]:
# Relação entre desconto e margem (média)
discount_analysis = (
    df.groupby("Discount")
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Orders=("Order ID", "count")
    )
    .assign(Margin=lambda x: x["Profit"] / x["Sales"])
    .sort_index()
)
print("\n--- Discount vs Margin ---")
print(discount_analysis)

# Análise mais clara: margem média por faixa de desconto
print("\n--- Discount vs Average Margin ---")
print(discount_analysis[["Sales", "Profit", "Margin", "Orders"]].round(3))

# Gráfico: Margem de Lucro por nível de Desconto
fig, ax = plt.subplots(figsize=(10, 5))
discount_analysis["Margin"].plot(kind="bar", ax=ax, color="#e67e22")
ax.set_title("Profit Margin by Discount Level", fontsize=14, pad=15)
ax.set_ylabel("Profit Margin")
ax.set_xlabel("Discount")
ax.axhline(0, color="black", linewidth=0.8)
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

# ============================================
# 8. TEMPORAL ANALYSIS
# ============================================

In [ ]:
yearly = (
    df.groupby("Order Year")[["Sales", "Profit"]]
    .sum()
    .assign(Margin=lambda x: x["Profit"] / x["Sales"])
)
print("\n--- Yearly Performance ---")
print(yearly)

monthly_sales = df.groupby("Order Month")["Sales"].sum()
# Gráfico melhorado - Vendas Mensais
fig, ax = plt.subplots(figsize=(12, 5))
monthly_sales.plot(kind="line", ax=ax, marker="o", linewidth=2, color="#2980b9")
ax.set_title("Monthly Sales Over Time", fontsize=14, pad=15)
ax.set_ylabel("Sales ($)")
ax.set_xlabel("Month")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================
# 9. REGIONAL ANALYSIS
# ============================================

In [ ]:
region = (
    df.groupby("Region")[["Sales", "Profit"]]
    .sum()
    .assign(Margin=lambda x: x["Profit"] / x["Sales"])
    .sort_values("Profit", ascending=False)
)
print("\n--- Region Performance ---")
print(region)

# ============================================
# 10. SUB-CATEGORY ANALYSIS
# ============================================

In [ ]:
subcategory = (
    df.groupby("Sub-Category")[["Sales", "Profit"]]
    .sum()
    .assign(Margin=lambda x: x["Profit"] / x["Sales"])
    .sort_values("Profit", ascending=False)
)
print("\n--- Sub-Category by Profit ---")
print(subcategory)

print("\n--- Sub-Category by Margin (pior para melhor) ---")
print(subcategory.sort_values("Margin"))

# Top 5 e Bottom 5 Sub-Categories por Lucro
top5 = subcategory.head(5)["Profit"]
bottom5 = subcategory.tail(5)["Profit"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top5.plot(kind="barh", ax=axes[0], color="#27ae60")
axes[0].set_title("Top 5 Sub-Categories by Profit")
axes[0].set_xlabel("Profit ($)")

bottom5.plot(kind="barh", ax=axes[1], color="#c0392b")
axes[1].set_title("Bottom 5 Sub-Categories by Profit")
axes[1].set_xlabel("Profit ($)")

plt.tight_layout()
plt.show()

# ============================================
# 11. KEY FINDINGS
# ============================================

In [ ]:
print("\n" + "="*60)
print("KEY FINDINGS")
print("="*60)

print(f"\nTotal Sales: ${total_sales:,.2f}")
print(f"Total Profit: ${total_profit:,.2f}")
print(f"Overall Margin: {overall_margin:.2%}")

print(f"\nBest Category by Profit: {category.index[0]}")
print(f"Worst Sub-Category by Margin: {subcategory.sort_values('Margin').index[0]}")
print(f"Best Region: {region.index[0]}")

# Segment
try:
    print(f"\nBest Segment by Profit: {segment.index[0]}")
except NameError:
    print("\nBest Segment by Profit: (ainda não calculado - rode a célula de Segment)")

# Ship Mode
try:
    print(f"Best Ship Mode by Profit: {ship_mode.index[0]}")
except NameError:
    print("Best Ship Mode by Profit: (ainda não calculado - rode a célula de Ship Mode)")

print(f"Average Shipping Days: {df['Shipping Days'].mean():.1f}")

## 12. Segment Analysis

Sales, profit and margin are compared across customer segments (Consumer, Corporate, Home Office).

In [ ]:
# Análise por Segment
segment = (
    df.groupby("Segment")[["Sales", "Profit"]]
    .sum()
    .assign(Margin=lambda x: x["Profit"] / x["Sales"])
    .sort_values("Profit", ascending=False)
)

print("--- Segment Performance ---")
print(segment.round(2))

# Gráfico
fig, ax = plt.subplots(figsize=(8, 5))
segment["Profit"].plot(kind="bar", ax=ax, color=["#3498db", "#2ecc71", "#9b59b6"])
ax.set_title("Profit by Customer Segment", fontsize=14, pad=15)
ax.set_ylabel("Profit ($)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
ax.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

## 13. Ship Mode Analysis

Comparison of sales, profit and average shipping days across different shipping modes.

In [ ]:
# Análise por Ship Mode
ship_mode = (
    df.groupby("Ship Mode")
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Orders=("Order ID", "count"),
        Avg_Shipping_Days=("Shipping Days", "mean")
    )
    .assign(Margin=lambda x: x["Profit"] / x["Sales"])
    .sort_values("Profit", ascending=False)
)

print("--- Ship Mode Performance ---")
print(ship_mode.round(2))

# Gráfico de lucro por modo de envio
fig, ax = plt.subplots(figsize=(9, 5))
ship_mode["Profit"].plot(kind="bar", ax=ax, color="#1abc9c")
ax.set_title("Profit by Ship Mode", fontsize=14, pad=15)
ax.set_ylabel("Profit ($)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=15)
ax.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

## 14. Shipping Days Analysis

Distribution of shipping time and its relationship with profitability.

In [ ]:
# Distribuição de dias de envio
print("--- Shipping Days Distribution ---")
print(df["Shipping Days"].describe().round(2))

print("\nValue counts (top values):")
print(df["Shipping Days"].value_counts().sort_index().head(10))

# Relação entre dias de envio e margem média
shipping_profit = (
    df.groupby("Shipping Days")
    .agg(
        Orders=("Order ID", "count"),
        Profit=("Profit", "sum"),
        Sales=("Sales", "sum")
    )
    .assign(Margin=lambda x: x["Profit"] / x["Sales"])
)

# Filtrar apenas dias com volume razoável de pedidos (evitar ruído)
shipping_profit = shipping_profit[shipping_profit["Orders"] >= 50]

print("\n--- Margin by Shipping Days (min 50 orders) ---")
print(shipping_profit.round(3))

# Gráfico
fig, ax = plt.subplots(figsize=(10, 5))
shipping_profit["Margin"].plot(kind="bar", ax=ax, color="#e67e22")
ax.set_title("Average Profit Margin by Shipping Days", fontsize=14, pad=15)
ax.set_ylabel("Profit Margin")
ax.set_xlabel("Shipping Days")
ax.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()